In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
print("="*60)
print("CREATING RELATIONSHIP DATASETS")
print("="*60)

# Define paths
processed_path = '../data/processed/'
output_path = '../data/relationship/'

# Create output directory
os.makedirs(output_path, exist_ok=True)

CREATING RELATIONSHIP DATASETS


In [3]:
# Load all cleaned files
crude_oil = pd.read_csv(os.path.join(processed_path, 'Crud_Oil_Price_clean.csv'))
petrol_diesel = pd.read_csv(os.path.join(processed_path, 'uk_petrol_diesel_clean.csv'))
fuel_inflation = pd.read_csv(os.path.join(processed_path, 'uk_cpih_fuel_energy_inflation_clean.csv'))
spending_by_income = pd.read_csv(os.path.join(processed_path, 'uk_household_spending_by_income_group_clean.csv'))
expenditure_by_decile = pd.read_csv(os.path.join(processed_path, 'uk_household_expenditure_by_decile_clean.csv'))
fuel_summary = pd.read_csv(os.path.join(processed_path, 'uk_fuel_summary_by_income_group_clean.csv'))
fuel_transport_decile = pd.read_csv(os.path.join(processed_path, 'uk_fuel_transport_spending_by_decile_clean.csv'))
petrol_decile = pd.read_csv(os.path.join(processed_path, 'uk_petrol_diesel_spending_by_decile_clean.csv'))

In [4]:
# Convert dates
crude_oil['Date'] = pd.to_datetime(crude_oil['Date'])
petrol_diesel['Date'] = pd.to_datetime(petrol_diesel['Date'])
fuel_inflation['Time'] = pd.to_datetime(fuel_inflation['Time'])

In [5]:
# Rename CPI columns for clarity
cpi_column_mapping = {
    '07.2.2 Fuels and lubricants': 'CPI_Fuels_and_Lubricants',
    '04.5.1 Electricity': 'CPI_Electricity',
    '04.5.2 Gas': 'CPI_Gas',
    '04.5.3 Liquid fuels': 'CPI_Liquid_Fuels',
    '04.5.4 Solid fuels': 'CPI_Solid_Fuels'
}

fuel_inflation.rename(columns=cpi_column_mapping, inplace=True)

print("✓ CPI columns renamed for clarity")

✓ CPI columns renamed for clarity


In [6]:
# Relationship R1 - Oil Price → Petrol/Diesel Price
print("\n" + "-"*40)
print("R1: Creating Oil → Petrol Relationship")
print("-"*40)

# Aggregate to yearly
crude_oil['Year'] = crude_oil['Date'].dt.year
oil_yearly = crude_oil.groupby('Year').agg({
    'Price': 'mean',
    'Open': 'mean',
    'High': 'mean',
    'Low': 'mean'
}).reset_index()

# Rename oil columns
oil_yearly.rename(columns={
    'Price': 'Crude_Oil_Price_USD',
    'Open': 'Crude_Oil_Open',
    'High': 'Crude_Oil_High',
    'Low': 'Crude_Oil_Low'
}, inplace=True)

petrol_diesel['Year'] = petrol_diesel['Date'].dt.year
fuel_yearly = petrol_diesel.groupby('Year').agg({
    'Petrol_Price_pence_per_litre': 'mean',
    'Diesel_Price_pence_per_litre': 'mean'
}).reset_index()

# Merge
r1_oil_petrol = pd.merge(oil_yearly, fuel_yearly, on='Year', how='inner')
r1_oil_petrol = r1_oil_petrol[r1_oil_petrol['Year'] >= 2020]

# Save
r1_oil_petrol.to_csv(os.path.join(output_path, 'r1_oil_to_petrol_relationship.csv'), index=False)
print(f"✓ Saved: r1_oil_to_petrol_relationship.csv ({len(r1_oil_petrol)} rows, {r1_oil_petrol['Year'].min()}-{r1_oil_petrol['Year'].max()})")
print("\nColumns now:")
print(r1_oil_petrol.columns.tolist())
print(r1_oil_petrol.head())



----------------------------------------
R1: Creating Oil → Petrol Relationship
----------------------------------------
✓ Saved: r1_oil_to_petrol_relationship.csv (6 rows, 2021-2026)

Columns now:
['Year', 'Crude_Oil_Price_USD', 'Crude_Oil_Open', 'Crude_Oil_High', 'Crude_Oil_Low', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre']
   Year  Crude_Oil_Price_USD  Crude_Oil_Open  Crude_Oil_High  Crude_Oil_Low  \
0  2021           531.979072      532.128603      533.544404     530.669550   
1  2022           573.121327      572.722041      575.848061     570.272041   
2  2023           548.526875      548.473750      549.827396     547.281562   
3  2024           585.843235      585.202255      588.209804     583.138725   
4  2025           549.077525      549.910792      551.933168     547.454455   

   Petrol_Price_pence_per_litre  Diesel_Price_pence_per_litre  
0                    121.320548                    130.772329  
1                    128.913973                  

In [7]:
# Relationship R2 - Fuel Prices → CPI Inflation
print("\n" + "-"*40)
print("R2: Creating Fuel → CPI Inflation Relationship")
print("-"*40)

fuel_inflation['Year'] = fuel_inflation['Time'].dt.year
cpi_yearly = fuel_inflation.groupby('Year').agg({
    'CPI_Fuels_and_Lubricants': 'mean',
    'CPI_Electricity': 'mean',
    'CPI_Gas': 'mean',
    'CPI_Liquid_Fuels': 'mean',
    'CPI_Solid_Fuels': 'mean'
}).reset_index()

# Merge with fuel prices
r2_fuel_cpi = pd.merge(fuel_yearly, cpi_yearly, on='Year', how='inner')
r2_fuel_cpi = r2_fuel_cpi[r2_fuel_cpi['Year'] >= 2020]

# Save
r2_fuel_cpi.to_csv(os.path.join(output_path, 'r2_fuel_to_cpi_relationship.csv'), index=False)
print(f"✓ Saved: r2_fuel_to_cpi_relationship.csv ({len(r2_fuel_cpi)} rows, {r2_fuel_cpi['Year'].min()}-{r2_fuel_cpi['Year'].max()})")
print("\nColumns now:")
print(r2_fuel_cpi.columns.tolist())
print(r2_fuel_cpi.head())



----------------------------------------
R2: Creating Fuel → CPI Inflation Relationship
----------------------------------------
✓ Saved: r2_fuel_to_cpi_relationship.csv (0 rows, nan-nan)

Columns now:
['Year', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'CPI_Fuels_and_Lubricants', 'CPI_Electricity', 'CPI_Gas', 'CPI_Liquid_Fuels', 'CPI_Solid_Fuels']
Empty DataFrame
Columns: [Year, Petrol_Price_pence_per_litre, Diesel_Price_pence_per_litre, CPI_Fuels_and_Lubricants, CPI_Electricity, CPI_Gas, CPI_Liquid_Fuels, CPI_Solid_Fuels]
Index: []


In [8]:
# Relationship R3 - Fuel Prices → Household Spending by Income Group
print("\n" + "-"*40)
print("R3: Creating Fuel → Household Spending (Income Group) Relationship")
print("-"*40)

# Pivot spending data
spending_pivot = spending_by_income.pivot_table(
    index=['Year', 'Category'],
    columns='Income_Group',
    values='Value'
).reset_index()
spending_pivot.columns.name = None

# Rename income columns for clarity
income_column_mapping = {
    'Lowest_20%': 'Spending_Lowest_20',
    'Second_20%': 'Spending_Second_20',
    'Middle_20%': 'Spending_Middle_20',
    'Fourth_20%': 'Spending_Fourth_20',
    'Highest_20%': 'Spending_Highest_20'
}

# Only rename columns that exist
rename_cols = {k: v for k, v in income_column_mapping.items() if k in spending_pivot.columns}
spending_pivot.rename(columns=rename_cols, inplace=True)

# Merge with fuel prices
r3_fuel_spending = pd.merge(spending_pivot, fuel_yearly, on='Year', how='left')
r3_fuel_spending = pd.merge(r3_fuel_spending, oil_yearly[['Year', 'Crude_Oil_Price_USD']], on='Year', how='left')
r3_fuel_spending = r3_fuel_spending[r3_fuel_spending['Year'] >= 2020]

# Save
r3_fuel_spending.to_csv(os.path.join(output_path, 'r3_fuel_to_household_spending.csv'), index=False)
print(f"✓ Saved: r3_fuel_to_household_spending.csv ({len(r3_fuel_spending)} rows)")
print(f"Categories: {r3_fuel_spending['Category'].unique().tolist()}")
print("\nColumns now:")
print(r3_fuel_spending.columns.tolist())
print(r3_fuel_spending.head())


----------------------------------------
R3: Creating Fuel → Household Spending (Income Group) Relationship
----------------------------------------
✓ Saved: r3_fuel_to_household_spending.csv (288 rows)
Categories: ['Alcoholic drinks', 'Audio-visual equipment', 'Beef', 'Bread, rice and cereals', 'Buns, cakes, biscuits', 'Cleaning materials', 'Clothing', 'Combined telecom', 'Electricity', 'Fish', 'Food', 'Footwear', 'Fruit', 'Furniture and furnishings', 'Gas', 'Hospital services', 'Hotel accommodation', 'Household appliances', 'Household hardware', 'Household textiles', 'Housing (net)', 'Insurance', 'Internet', 'Maintenance and repair', 'Medical products', 'Milk and dairy', 'Newspapers and books', 'Non-alcoholic drinks', 'Other food', 'Other fuels', 'Other services', 'Package holidays', 'Personal care', 'Petrol, diesel and oils', 'Pets and pet food', 'Poultry', 'Public transport', 'Restaurant meals', 'Sports and recreation', 'Sugar and confectionery', 'TV subscriptions', 'Take-away foo

In [9]:
# Relationship R4 - Fuel Prices → Detailed Decile Spending
print("\n" + "-"*40)
print("R4: Creating Fuel → Decile Spending Relationship")
print("-"*40)

# Pivot decile data
decile_pivot = expenditure_by_decile.pivot_table(
    index=['Year', 'Category'],
    columns='Decile',
    values='Value'
).reset_index()
decile_pivot.columns.name = None

# Rename decile columns
decile_column_mapping = {
    'Lowest_10%': 'Spending_Decile_1',
    'Second_10%': 'Spending_Decile_2',
    'Third_10%': 'Spending_Decile_3',
    'Fourth_10%': 'Spending_Decile_4',
    'Fifth_10%': 'Spending_Decile_5',
    'Sixth_10%': 'Spending_Decile_6',
    'Seventh_10%': 'Spending_Decile_7',
    'Eighth_10%': 'Spending_Decile_8',
    'Ninth_10%': 'Spending_Decile_9',
    'Highest_10%': 'Spending_Decile_10'
}

# Only rename columns that exist
rename_decile = {k: v for k, v in decile_column_mapping.items() if k in decile_pivot.columns}
decile_pivot.rename(columns=rename_decile, inplace=True)

# Merge with fuel prices
r4_fuel_decile = pd.merge(decile_pivot, fuel_yearly, on='Year', how='left')
r4_fuel_decile = pd.merge(r4_fuel_decile, oil_yearly[['Year', 'Crude_Oil_Price_USD']], on='Year', how='left')
r4_fuel_decile = r4_fuel_decile[r4_fuel_decile['Year'] >= 2020]

# Save
r4_fuel_decile.to_csv(os.path.join(output_path, 'r4_fuel_to_decile_spending.csv'), index=False)
print(f"✓ Saved: r4_fuel_to_decile_spending.csv ({len(r4_fuel_decile)} rows)")
print(f"Categories: {r4_fuel_decile['Category'].unique().tolist()}")
print("\nColumns now:")
print(r4_fuel_decile.columns.tolist()[:10])  # First 10 columns
print(r4_fuel_decile.head())


----------------------------------------
R4: Creating Fuel → Decile Spending Relationship
----------------------------------------
✓ Saved: r4_fuel_to_decile_spending.csv (72 rows)
Categories: ['Alcoholic drink, tobacco & narcotics', 'Clothing & footwear', 'Communication', 'Education', 'Food & non-alcoholic drinks', 'Health', 'Household goods & services', 'Housing (net), fuel & power', 'Miscellaneous goods & services', 'Recreation & culture', 'Restaurants & hotels', 'Transport']

Columns now:
['Year', 'Category', 'Spending_Decile_8', 'Spending_Decile_5', 'Spending_Decile_4', 'Spending_Decile_10', 'Spending_Decile_1', 'Spending_Decile_9', 'Spending_Decile_2', 'Spending_Decile_7']
   Year                              Category  Spending_Decile_8  \
0  2020  Alcoholic drink, tobacco & narcotics              16.65   
1  2020                   Clothing & footwear              29.84   
2  2020                         Communication              27.03   
3  2020                             Edu

In [10]:
# Relationship R5 - Fuel Prices → Fuel-Specific Spending (by Income Group)
print("\n" + "-"*40)
print("R5: Creating Fuel → Fuel-Specific Spending Relationship")
print("-"*40)

# Pivot fuel summary
fuel_summary_pivot = fuel_summary.pivot_table(
    index=['Year', 'Category'],
    columns='Income_Group',
    values='Value'
).reset_index()
fuel_summary_pivot.columns.name = None

# Rename income columns
rename_fuel_summary = {k: v for k, v in income_column_mapping.items() if k in fuel_summary_pivot.columns}
fuel_summary_pivot.rename(columns=rename_fuel_summary, inplace=True)

# Merge with fuel prices
r5_fuel_specific = pd.merge(fuel_summary_pivot, fuel_yearly, on='Year', how='left')
r5_fuel_specific = pd.merge(r5_fuel_specific, oil_yearly[['Year', 'Crude_Oil_Price_USD']], on='Year', how='left')
r5_fuel_specific = r5_fuel_specific[r5_fuel_specific['Year'] >= 2020]

# Save
r5_fuel_specific.to_csv(os.path.join(output_path, 'r5_fuel_specific_spending.csv'), index=False)
print(f"✓ Saved: r5_fuel_specific_spending.csv ({len(r5_fuel_specific)} rows)")
print("\nColumns now:")
print(r5_fuel_specific.columns.tolist())
print(r5_fuel_specific.head())


----------------------------------------
R5: Creating Fuel → Fuel-Specific Spending Relationship
----------------------------------------
✓ Saved: r5_fuel_specific_spending.csv (24 rows)

Columns now:
['Year', 'Category', 'Spending_Highest_20', 'Spending_Lowest_20', 'Middle_40%', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'Crude_Oil_Price_USD']
   Year                 Category  Spending_Highest_20  Spending_Lowest_20  \
0  2020              Electricity                20.15                6.27   
1  2020                      Gas                16.96                5.22   
2  2020              Other fuels                 2.50                0.76   
3  2020  Petrol, diesel and oils                35.02               10.90   
4  2021              Electricity                20.74                6.61   

   Middle_40%  Petrol_Price_pence_per_litre  Diesel_Price_pence_per_litre  \
0       11.92                    116.000000                    124.700000   
1        9.62 

In [11]:
# Relationship R6 - Fuel Prices → Petrol/Diesel Spending by Decile
print("\n" + "-"*40)
print("R6: Creating Fuel → Petrol/Diesel Spending by Decile Relationship")
print("-"*40)

# Pivot petrol decile
petrol_decile_pivot = petrol_decile.pivot_table(
    index=['Year', 'Category'],
    columns='Decile',
    values='Value'
).reset_index()
petrol_decile_pivot.columns.name = None

# Rename decile columns
rename_petrol_decile = {k: v for k, v in decile_column_mapping.items() if k in petrol_decile_pivot.columns}
petrol_decile_pivot.rename(columns=rename_petrol_decile, inplace=True)

# Merge with fuel prices
r6_petrol_decile = pd.merge(petrol_decile_pivot, fuel_yearly, on='Year', how='left')
r6_petrol_decile = pd.merge(r6_petrol_decile, oil_yearly[['Year', 'Crude_Oil_Price_USD']], on='Year', how='left')
r6_petrol_decile = r6_petrol_decile[r6_petrol_decile['Year'] >= 2020]

# Save
r6_petrol_decile.to_csv(os.path.join(output_path, 'r6_petrol_decile_spending.csv'), index=False)
print(f"✓ Saved: r6_petrol_decile_spending.csv ({len(r6_petrol_decile)} rows)")
print("\nColumns now:")
print(r6_petrol_decile.columns.tolist())
print(r6_petrol_decile.head())


----------------------------------------
R6: Creating Fuel → Petrol/Diesel Spending by Decile Relationship
----------------------------------------
✓ Saved: r6_petrol_decile_spending.csv (18 rows)

Columns now:
['Year', 'Category', 'Spending_Decile_8', 'Spending_Decile_5', 'Spending_Decile_4', 'Spending_Decile_10', 'Spending_Decile_1', 'Spending_Decile_9', 'Spending_Decile_2', 'Spending_Decile_7', 'Spending_Decile_6', 'Spending_Decile_3', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'Crude_Oil_Price_USD']
   Year          Category  Spending_Decile_8  Spending_Decile_5  \
0  2020            Diesel               9.40               6.28   
1  2020  Other motor oils               1.02               0.68   
2  2020            Petrol              19.51              12.47   
3  2021            Diesel              10.70               7.11   
4  2021  Other motor oils               1.20               0.82   

   Spending_Decile_4  Spending_Decile_10  Spending_Decile_1  \
0  

In [12]:
# Relationship R7 - Fuel Prices → Transport-Specific Spending by Decile
print("\n" + "-"*40)
print("R7: Creating Fuel → Transport Spending by Decile Relationship")
print("-"*40)

# Pivot fuel transport decile
transport_decile_pivot = fuel_transport_decile.pivot_table(
    index=['Year', 'Category'],
    columns='Decile',
    values='Value'
).reset_index()
transport_decile_pivot.columns.name = None

# Rename decile columns
rename_transport_decile = {k: v for k, v in decile_column_mapping.items() if k in transport_decile_pivot.columns}
transport_decile_pivot.rename(columns=rename_transport_decile, inplace=True)

# Merge with fuel prices
r7_transport_decile = pd.merge(transport_decile_pivot, fuel_yearly, on='Year', how='left')
r7_transport_decile = pd.merge(r7_transport_decile, oil_yearly[['Year', 'Crude_Oil_Price_USD']], on='Year', how='left')
r7_transport_decile = r7_transport_decile[r7_transport_decile['Year'] >= 2020]

# Save
r7_transport_decile.to_csv(os.path.join(output_path, 'r7_transport_decile_spending.csv'), index=False)
print(f"✓ Saved: r7_transport_decile_spending.csv ({len(r7_transport_decile)} rows)")
print("\nColumns now:")
print(r7_transport_decile.columns.tolist())
print(r7_transport_decile.head())


----------------------------------------
R7: Creating Fuel → Transport Spending by Decile Relationship
----------------------------------------
✓ Saved: r7_transport_decile_spending.csv (42 rows)

Columns now:
['Year', 'Category', 'Spending_Decile_8', 'Spending_Decile_5', 'Spending_Decile_4', 'Spending_Decile_10', 'Spending_Decile_1', 'Spending_Decile_9', 'Spending_Decile_2', 'Spending_Decile_7', 'Spending_Decile_6', 'Spending_Decile_3', 'Petrol_Price_pence_per_litre', 'Diesel_Price_pence_per_litre', 'Crude_Oil_Price_USD']
   Year                 Category  Spending_Decile_8  Spending_Decile_5  \
0  2020              Electricity              15.61              10.69   
1  2020                      Gas              13.58               8.98   
2  2020              Other fuels               1.95               1.28   
3  2020  Petrol, diesel and oils              27.86              19.85   
4  2020         Public transport               8.90               6.08   

   Spending_Decile_4  Spe

In [13]:
# Create MASTER TABLE - Unified Relationship Dataset
print("\n" + "="*60)
print("CREATING MASTER RELATIONSHIP TABLE")
print("="*60)

# Start with R3 (household spending by income group)
master_table = r3_fuel_spending.copy()

# Add CPI inflation data with clean names
master_table = pd.merge(master_table, cpi_yearly, on='Year', how='left')

# Reorder columns for clarity with clean names
column_order = [
    'Year', 
    'Category',
    'Spending_Lowest_20', 
    'Spending_Second_20',
    'Spending_Middle_20', 
    'Spending_Fourth_20',
    'Spending_Highest_20',
    'Crude_Oil_Price_USD',
    'Petrol_Price_pence_per_litre',
    'Diesel_Price_pence_per_litre',
    'CPI_Fuels_and_Lubricants',
    'CPI_Electricity',
    'CPI_Gas',
    'CPI_Liquid_Fuels',
    'CPI_Solid_Fuels'
]

# Only include columns that exist
existing_cols = [col for col in column_order if col in master_table.columns]
master_table = master_table[existing_cols]

# Sort
master_table = master_table.sort_values(['Category', 'Year']).reset_index(drop=True)

# Save master table
master_table.to_csv(os.path.join(output_path, 'master_relationship_table.csv'), index=False)
print(f"✓ Saved: master_relationship_table.csv")
print(f"  Shape: {master_table.shape}")
print(f"  Years: {master_table['Year'].min()} - {master_table['Year'].max()}")
print(f"  Categories: {master_table['Category'].nunique()}")


CREATING MASTER RELATIONSHIP TABLE
✓ Saved: master_relationship_table.csv
  Shape: (288, 15)
  Years: 2020 - 2025
  Categories: 48


In [14]:
# Create Summary of All Relationship Datasets
print("\n" + "="*60)
print("RELATIONSHIP DATASETS SUMMARY")
print("="*60)

relationship_summary = pd.DataFrame({
    'Dataset': [
        'r1_oil_to_petrol_relationship',
        'r2_fuel_to_cpi_relationship',
        'r3_fuel_to_household_spending',
        'r4_fuel_to_decile_spending',
        'r5_fuel_specific_spending',
        'r6_petrol_decile_spending',
        'r7_transport_decile_spending',
        'master_relationship_table'
    ],
    'Shape': [
        r1_oil_petrol.shape,
        r2_fuel_cpi.shape,
        r3_fuel_spending.shape,
        r4_fuel_decile.shape,
        r5_fuel_specific.shape,
        r6_petrol_decile.shape,
        r7_transport_decile.shape,
        master_table.shape
    ],
    'Columns': [
        len(r1_oil_petrol.columns),
        len(r2_fuel_cpi.columns),
        len(r3_fuel_spending.columns),
        len(r4_fuel_decile.columns),
        len(r5_fuel_specific.columns),
        len(r6_petrol_decile.columns),
        len(r7_transport_decile.columns),
        len(master_table.columns)
    ]
})

print(relationship_summary.to_string(index=False))


RELATIONSHIP DATASETS SUMMARY
                      Dataset     Shape  Columns
r1_oil_to_petrol_relationship    (6, 7)        7
  r2_fuel_to_cpi_relationship    (0, 8)        8
r3_fuel_to_household_spending (288, 10)       10
   r4_fuel_to_decile_spending  (72, 15)       15
    r5_fuel_specific_spending   (24, 8)        8
    r6_petrol_decile_spending  (18, 15)       15
 r7_transport_decile_spending  (42, 15)       15
    master_relationship_table (288, 15)       15
